In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!pip install ultralytics roboflow -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 110.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 5.5 MB/s eta 0:00:00


In [9]:
from getpass import getpass
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key=getpass("Roboflow API key: "))
project = rf.workspace("pagamos-claude").project("microbiologia")
version = project.version(1)
dataset = version.download("yolov11")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to microbiologia-1 in yolov11:: 100%|██████████| 7199/7199 [00:01<00:00, 4172.46it/s]


In [10]:
import os
import glob

def convert_polygon_to_bbox(values):
    xs = values[0::2]
    ys = values[1::2]
    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)
    x_center = (x_min + x_max) / 2
    y_center = (y_min + y_max) / 2
    width = x_max - x_min
    height = y_max - y_min
    return [x_center, y_center, width, height]

label_dirs = [
    f"{dataset.location}/train/labels",
    f"{dataset.location}/valid/labels",
    f"{dataset.location}/test/labels",
]

fixed_files = 0
fixed_lines = 0

for label_dir in label_dirs:
    if not os.path.exists(label_dir):
        continue
    for filepath in glob.glob(os.path.join(label_dir, "*.txt")):
        with open(filepath, "r") as f:
            lines = f.readlines()

        new_lines = []
        file_changed = False

        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
            class_id = parts[0]
            values = [float(v) for v in parts[1:]]

            if len(values) == 4:
                new_lines.append(line.strip())
            elif len(values) > 4 and len(values) % 2 == 0:
                bbox = convert_polygon_to_bbox(values)
                new_line = f"{class_id} " + " ".join(f"{v:.6f}" for v in bbox)
                new_lines.append(new_line)
                file_changed = True
                fixed_lines += 1
            else:
                print(f"Línea con formato inesperado en {filepath}: {line.strip()}")

        if file_changed:
            with open(filepath, "w") as f:
                f.write("\n".join(new_lines) + "\n")
            fixed_files += 1

print(f"Archivos corregidos (polígono→bbox): {fixed_files}")
print(f"Líneas convertidas: {fixed_lines}")


Archivos corregidos (polígono→bbox): 2826
Líneas convertidas: 2900


In [11]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")  # solo para leer model.names con las clases del data.yaml; no entrena nada todavía

# Verifica los IDs antes de aplicar nada
import yaml
with open(f"{dataset.location}/data.yaml") as f:
    data_yaml = yaml.safe_load(f)
class_names_list = data_yaml['names']
print(class_names_list)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
['Agitador Orbital JOANLAB OS-20', 'Autoclave ALL AMERICAN 25X-1', 'Balanza analitica PR Series Analytical', 'Bano maria  Memmert WNB-14', 'Cabina de flujo laminar horizontal -PIVAS- BBS-H1500B BBS-H1800B', 'Centrifugadora Sigma 201', 'Espectofotometro UV-5100B', 'Estereo Microscopio Binocular Modelo BS-80', 'Estereo Microscopio Thomas Scientific', 'Estufa de secado memmert ULE 600', 'Incubadora de laboratorio SMI6', 'Microscopio Motic RED 220', 'Microscopio Olympus CX22 LED', 'incubadora memmert in110']


In [12]:
estufa_id = 9
bs80_id = 7

burst_pattern = "151610_BURST"
splits = ["train", "valid", "test"]
found_files = []

for split in splits:
    label_dir = f"{dataset.location}/{split}/labels"
    if not os.path.exists(label_dir):
        continue
    for filepath in glob.glob(os.path.join(label_dir, "*.txt")):
        if burst_pattern in os.path.basename(filepath):
            found_files.append(filepath)

print(f"Archivos del burst encontrados: {len(found_files)}")

Archivos del burst encontrados: 85


In [13]:
fixed = 0
for filepath in found_files:
    with open(filepath, "r") as f:
        lines = f.readlines()

    new_lines = []
    changed = False
    for line in lines:
        parts = line.strip().split()
        cls_id = int(parts[0])
        if cls_id == estufa_id:
            parts[0] = str(bs80_id)
            changed = True
        new_lines.append(" ".join(parts))

    if changed:
        with open(filepath, "w") as f:
            f.write("\n".join(new_lines) + "\n")
        fixed += 1

print(f"Archivos corregidos (clase Estufa→BS-80): {fixed}")

Archivos corregidos (clase Estufa→BS-80): 10


In [14]:
# 1. Confirma que ya no quedan líneas mal formadas (polígono) en ningún label
problematic = 0
for label_dir in label_dirs:
    for filepath in glob.glob(os.path.join(label_dir, "*.txt")):
        with open(filepath) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    problematic += 1
                    print(f"Aún mal formada: {filepath}")
print(f"Líneas con formato incorrecto restantes: {problematic}")  # debe dar 0

# 2. Confirma que ya no queda ninguna instancia de clase Estufa en el burst 151610
still_wrong = 0
for filepath in found_files:
    with open(filepath) as f:
        for line in f:
            if int(line.split()[0]) == estufa_id:
                still_wrong += 1
print(f"Instancias de Estufa restantes en el burst: {still_wrong}")  # debe dar 0

# 3. Confirma que no hay imágenes corruptas/vacías
empty_labels = 0
for label_dir in label_dirs:
    for filepath in glob.glob(os.path.join(label_dir, "*.txt")):
        if os.path.getsize(filepath) == 0:
            empty_labels += 1
print(f"Archivos de label vacíos: {empty_labels}")  # informativo, no necesariamente un error

Líneas con formato incorrecto restantes: 0
Instancias de Estufa restantes en el burst: 0
Archivos de label vacíos: 0


In [15]:
# Verificación final explícita, sin reutilizar variables de pasos anteriores
import glob, os

total_lines_bad_format = 0
total_estufa_in_burst = 0
total_files_checked = 0

for split in ["train", "valid", "test"]:
    label_dir = f"{dataset.location}/{split}/labels"
    for filepath in glob.glob(os.path.join(label_dir, "*.txt")):
        total_files_checked += 1
        with open(filepath) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    total_lines_bad_format += 1
                if "151610_BURST" in os.path.basename(filepath) and int(parts[0]) == 9:
                    total_estufa_in_burst += 1

print(f"Total archivos revisados: {total_files_checked}")
print(f"Líneas mal formadas: {total_lines_bad_format}")
print(f"Instancias de Estufa aún en el burst: {total_estufa_in_burst}")

Total archivos revisados: 3597
Líneas mal formadas: 0
Instancias de Estufa aún en el burst: 0


In [16]:
model = YOLO("yolo11n.pt")
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    name="microbiologia_v1"  # o "microbiologia_v4" si preferís llevar la cuenta global
)

Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/microbiologia-1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=microbiologia_v1, nbs=64, n

In [17]:
import os
print(os.path.exists("/content/runs/detect/microbiologia_v1/weights/best.pt"))

True


In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
!mkdir -p /content/drive/MyDrive/microbiologia_proyecto
!cp -r /content/runs/detect/microbiologia_v1 /content/drive/MyDrive/microbiologia_proyecto/

In [20]:
saved_path = "/content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best.pt"
print(os.path.exists(saved_path))
print(os.path.getsize(saved_path) / (1024*1024), "MB")

True
5.223535537719727 MB


In [22]:
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics -q

from ultralytics import YOLO
model = YOLO("/content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best.pt")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
!cp -r /content/drive/MyDrive/microbiologia_proyecto/dataset_corregido /content/microbiologia-1

In [24]:
model.export(format="tflite", int8=True, data="/content/microbiologia-1/data.yaml")

WARNING ⚠️ 'int8' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ format='tflite' is deprecated as of 8.4.83 and has been replaced by the unified Google LiteRT format. Exporting format='litert' instead. See https://docs.ultralytics.com/integrations/litert
Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ LiteRT INT8 export does not support end2end models, disabling end2end branch.
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO11n summary (fused): 101 layers, 2,584,882 parameters, 0 gradients, 6.4 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 18, 8400) (5.2 MB)
LiteRT: collecting INT8 calibration images from 'data=/content/microbiologia-1/data.yaml'
val: Fast image access ✅ (ping:

invalid escape sequence '\.'



LiteRT: starting export with litert_torch 0.9.4...


(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

...

(00:00) [ DONE] Write Model to 
/content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best_int8.tflite (+00:00)

LiteRT: applying static quantization (int8 weights + int8 activations)...


Applying Transformations to tensors:: 100%|██████████| 572/572 [00:00<00:00, 12851.90it/s]


Model name: /content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best_int8.tflite
Original model size: 10.14 MiB
Quantized model size: 2.90 MiB
Quantization Ratio: 0.29 (3.5x smaller)
Total time: 319.07 ms
LiteRT: export success ✅ 266.1s, saved as '/content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best_int8.tflite' (2.9 MB)

Export complete (266.7s)
Results saved to /content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best_int8.tflite
Predict:         yolo predict task=detect model=/content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best_int8.tflite imgsz=640 
Validate:        yolo val task=detect model=/content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best_int8.tflite imgsz=640 data=/content/microbiologia-1/data.yaml  
Visualize:       https://netron.app


PosixPath('/content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best_int8.tflite')

In [25]:
model.export(format="tflite", data="/content/microbiologia-1/data.yaml")

WARNING ⚠️ format='tflite' is deprecated as of 8.4.83 and has been replaced by the unified Google LiteRT format. Exporting format='litert' instead. See https://docs.ultralytics.com/integrations/litert
Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
YOLO11n summary (fused): 101 layers, 2,584,882 parameters, 0 gradients, 6.4 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 18, 8400) (5.2 MB)

LiteRT: starting export with litert_torch 0.9.4...


(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions

`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

(00:13) [ DONE] LiteRT-Torch Convert (+00:13)

(00:00) [START] Write Model to /content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best.tflite

(00:00) [ DONE] Write Model to /content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best.tflite 
(+00:00)

LiteRT: export success ✅ 13.3s, saved as '/content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best.tflite' (10.1 MB)

Export complete (13.9s)
Results saved to /content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best.tflite
Predict:         yolo predict task=detect model=/content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best.tflite imgsz=640 
Validate:        yolo val task=detect model=/content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best.tflite imgsz=640 data=/content/microbiologia-1/data.yaml  
Visualize:       https://netron.app


PosixPath('/content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best.tflite')

In [27]:
model = YOLO("/content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best.pt")
result_path = model.export(format="tflite", int8=True, data="/content/microbiologia-1/data.yaml")
print("Exportado en:", result_path)

WARNING ⚠️ 'int8' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ format='tflite' is deprecated as of 8.4.83 and has been replaced by the unified Google LiteRT format. Exporting format='litert' instead. See https://docs.ultralytics.com/integrations/litert
Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ LiteRT INT8 export does not support end2end models, disabling end2end branch.
YOLO11n summary (fused): 101 layers, 2,584,882 parameters, 0 gradients, 6.4 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 18, 8400) (5.2 MB)
LiteRT: collecting INT8 calibration images from 'data=/content/microbiologia-1/data.yaml'
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1430.2±306.4 MB/s, size: 48.4 KB)
val: Scanning /content/microbiologia-1/valid/labels.cache... 300 images, 0 backgrounds, 

(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

(00:01) [START] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions

`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

(00:00) [START] Write Model to 
/content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best_int8.tflite

(00:00) [ DONE] Write Model to 
/content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best_int8.tflite (+00:00)

LiteRT: applying static quantization (int8 weights + int8 activations)...


Applying Transformations to tensors:: 100%|██████████| 572/572 [00:00<00:00, 26438.86it/s]


Model name: /content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best_int8.tflite
Original model size: 10.14 MiB
Quantized model size: 2.90 MiB
Quantization Ratio: 0.29 (3.5x smaller)
Total time: 184.45 ms
LiteRT: export success ✅ 245.9s, saved as '/content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best_int8.tflite' (2.9 MB)

Export complete (246.5s)
Results saved to /content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best_int8.tflite
Predict:         yolo predict task=detect model=/content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best_int8.tflite imgsz=640 
Validate:        yolo val task=detect model=/content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best_int8.tflite imgsz=640 data=/content/microbiologia-1/data.yaml  
Visualize:       https://netron.app
Exportado en: /content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best_int8.tflite


In [28]:
!find /content -iname "*best_saved_model*" 2>/dev/null
!find /content/drive -iname "*best_saved_model*" 2>/dev/null

In [29]:
!find / -iname "*.tflite" 2>/dev/null

/content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best_int8.tflite
/content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best.tflite


In [30]:
import os

weights_dir = "/content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/"

for f in ["best.tflite", "best_int8.tflite"]:
    path = os.path.join(weights_dir, f)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024*1024)
        print(f"{f}: {size_mb:.2f} MB")
    else:
        print(f"{f}: no encontrado")

best.tflite: 10.14 MB
best_int8.tflite: 2.90 MB


In [33]:
from ultralytics import YOLO

model_int8 = YOLO(os.path.join(weights_dir, "best_int8.tflite"))
metrics_int8 = model_int8.val(data="/content/microbiologia-1/data.yaml")
print("mAP50 (int8):", metrics_int8.box.map50)
print("mAP50-95 (int8):", metrics_int8.box.map)

Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Loading /content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best_int8.tflite for LiteRT inference...
Setting batch=1 input of shape (1, 3, 640, 640)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2217.2±719.3 MB/s, size: 53.0 KB)
val: Scanning /content/microbiologia-1/valid/labels.cache... 300 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 300/300 96.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 300/300 6.8it/s 44.0s
                   all        300        308      0.971      0.965      0.967      0.765
Agitador Orbital JOANLAB OS-20         28         30      0.928        0.9      0.894      0.537
Autoclave ALL AMERICAN 25X-1         25         26       0.99      0.923      0.925      0.861
Balanza analitica PR Series Analytical         23         23      0.974      0.957      0.955      0.612
Bano mar

In [32]:
from google.colab import files
files.download(os.path.join(weights_dir, "best_int8.tflite"))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
from google.colab import files
files.download("/content/drive/MyDrive/microbiologia_proyecto/microbiologia_v1/weights/best_int8.tflite")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
import glob, json, os, datetime

CLAVES = ['YOLO(', '.train(', '.export(', 'tflite', 'roboflow', 'ultralytics', 'data.yaml']

for nb in sorted(glob.glob('/content/drive/MyDrive/Colab Notebooks/*.ipynb')):
    try:
        celdas = json.load(open(nb, encoding='utf-8'))['cells']
    except Exception as e:
        print(f'{os.path.basename(nb)}: no se pudo leer ({e})')
        continue

    codigo = '\n'.join(''.join(c.get('source', ''))
                       for c in celdas if c.get('cell_type') == 'code')
    encontradas = [k for k in CLAVES if k.lower() in codigo.lower()]
    mtime = datetime.datetime.fromtimestamp(os.path.getmtime(nb))

    print(f'{os.path.basename(nb):<20} {len(celdas):>3} celdas  '
          f'{mtime:%Y-%m-%d %H:%M}  {os.path.getsize(nb)/1024:>6.0f} KB')
    print(f'   claves: {encontradas if encontradas else "ninguna"}')

Untitled0.ipynb        9 celdas  2026-09-03 14:09      33 KB
   claves: ['YOLO(', '.train(', 'roboflow', 'ultralytics', 'data.yaml']
Untitled1.ipynb        9 celdas  2026-09-04 00:18      47 KB
   claves: ['YOLO(', '.train(', 'roboflow', 'ultralytics', 'data.yaml']
Untitled2.ipynb        9 celdas  2026-09-04 00:31      25 KB
   claves: ['YOLO(', '.train(', 'roboflow', 'ultralytics', 'data.yaml']
Untitled3.ipynb       26 celdas  2026-09-04 06:16     163 KB
   claves: ['YOLO(', '.train(', '.export(', 'tflite', 'roboflow', 'ultralytics', 'data.yaml']
